In [14]:
from dotenv import load_dotenv
import requests
import os
import polars as pl

load_dotenv(override=True)

True

In [3]:
SITE_ID = os.environ["SOLAREDGE_SITE_ID"]
API_KEY = os.environ["SOLAREDGE_KEY"]

In [5]:
BASE = "https://monitoringapi.solaredge.com"
url  = f"{BASE}/site/{SITE_ID}/energy"

In [21]:
def get_data(url, **params):
    r = requests.get(url, params=params, timeout=30)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        # Print the exact server response (first 4k to be safe) before raising
        body = (r.text or "").strip()
        print("\n--- SolarEdge API Error -------------------------------------")
        print("Status:", r.status_code)
        print("URL   :", r.url)
        print("Body  :", body[:4096])
        print("--------------------------------------------------------------\n")
        raise

    try:
         result = r.json()
    except ValueError:
        print("\n--- Unexpected non-JSON response ----------------------------")
        print("URL   :", r.url)
        print("Body  :", (r.text or "")[:4096])
        print("--------------------------------------------------------------\n")
        raise

    energy = result["energy"]
    unit = energy["unit"]
    vals = energy["values"]

    df = pl.from_records(vals)

    # Parse 'date' as timezone-aware datetime in US/Eastern
    df = df.with_columns(
        pl.col("date")
        .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False)
        .dt.date()        # Convert to a date from a timestamp
    )

    # Add constant column for unit
    df = df.with_columns(pl.lit(unit).alias("unit"))
    return df

In [23]:
annual_dfs = []
for year in [str(y) for y in [2020, 2021, 2022, 2023, 2024, 2025]]:
    params = {"timeUnit": "DAY", "startDate": f"{year}-01-01", "endDate": f"{year}-12-31", "api_key": API_KEY}
    annual_dfs.append(get_data(url, **params))

all_time = pl.concat(annual_dfs)
all_time.write_parquet("solar_panels_daily.parquet")


In [25]:
all_time.head()

date,value,unit
date,f64,str
2020-01-01,13021.0,"""Wh"""
2020-01-02,13001.0,"""Wh"""
2020-01-03,6495.0,"""Wh"""
2020-01-04,2673.0,"""Wh"""
2020-01-05,11098.0,"""Wh"""


In [27]:
all_time.select(
    pl.col("value").max().alias("max"),
    pl.col("value").min().alias("min"),
)

max,min
f64,f64
82754.0,0.0
